# `sparsegf2.circuits.scheduler` - config plus seed to layers

`CircuitBuilder` is simulator-free. It converts a validated `CircuitConfig`
and one `sample_seed` into a deterministic stream of `CircuitLayer` records.
Each layer contains gate pairs, one symplectic Clifford-table index per gate,
the measurement candidates, and the subset that actually fires.

Schema v2 seeds the construction generator with the pair
`[base_seed, sample_seed]`. The old scalar sum aliased distinct cells such as
`(10, 2)` and `(11, 1)`; pair seeding keeps those streams independent.


In [1]:
from sparsegf2.circuits import CircuitBuilder, CircuitConfig, from_spec

cfg = CircuitConfig(
    graph_spec='cycle', n=8, p=0.25, total_layers_override=2,
    base_seed=10,
)
layer = CircuitBuilder(cfg, sample_seed=2).schedule()[0]
print('gate pairs       :', layer.gate_pairs)
print('Clifford indices :', layer.cliff_indices)
print('meas candidates  :', layer.meas_candidates)
print('meas qubits      :', layer.meas_qubits)


gate pairs       : [(0, 1), (2, 3), (4, 5), (6, 7)]
Clifford indices : [668 196 239 500]
meas candidates  : [0, 1, 2, 3, 4, 5, 6, 7]
meas qubits      : [1, 6]


## Four gate-placement modes

The fixed per-layer draw order is load-bearing for reproducibility:

1. place gates (`all_edges` and round-robin brickwork consume no placement
   randomness);
2. draw one Clifford index per gate;
3. select measurement candidates where needed, then draw their Bernoulli
   firing coins.

`random_edge` draws distinct edge indices. `random_pool` draws with replacement,
so repeated edges are allowed and the requested count may exceed `|E|`.
`all_edges` returns the complete stored edge list in its canonical order.


In [2]:
modes = {
    'brickwork': {},
    'random_edge': {'gates_per_layer': 2},
    'random_pool': {'gates_per_layer': 6},
    'all_edges': {},
}
for mode, extra in modes.items():
    local = CircuitConfig(
        graph_spec='cycle', n=8, gating_mode=mode, p=0.0,
        total_layers_override=1, **extra,
    )
    first = CircuitBuilder(local, 4).schedule()[0]
    print(f'{mode:>11}: {first.n_gates} gates  {first.gate_pairs}')
    if mode == 'all_edges':
        assert first.gate_pairs == from_spec('cycle', 8).edges


  brickwork: 4 gates  [(0, 1), (2, 3), (4, 5), (6, 7)]
random_edge: 2 gates  [(1, 2), (4, 5)]
random_pool: 6 gates  [(2, 3), (4, 5), (6, 7), (1, 2), (5, 6), (6, 7)]
  all_edges: 8 gates  [(0, 1), (0, 7), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7)]


## Four measurement modes

`bernoulli` makes every system qubit eligible. `gated` uses the distinct gate
endpoints. `random_pair` selects two distinct candidates, and `uniform_count`
selects `meas_count` distinct candidates. Candidate selection and firing are
recorded separately so circuit visualizations can show both.


In [3]:
measurement_modes = {
    'bernoulli': {},
    'gated': {},
    'random_pair': {},
    'uniform_count': {'meas_count': 3},
}
for mode, extra in measurement_modes.items():
    local = CircuitConfig(
        graph_spec='cycle', n=8, measurement_mode=mode, p=1.0,
        total_layers_override=1, **extra,
    )
    first = CircuitBuilder(local, 5).schedule()[0]
    assert first.meas_qubits == first.meas_candidates
    print(f'{mode:>13}: candidates={first.meas_candidates}')


    bernoulli: candidates=[0, 1, 2, 3, 4, 5, 6, 7]
        gated: candidates=[0, 1, 2, 3, 4, 5, 6, 7]
  random_pair: candidates=[0, 4]
uniform_count: candidates=[0, 4, 7]


## Pair-seeded construction and independent runner streams

Equal scalar sums no longer imply equal construction streams. The runner adds
two independent tagged streams without perturbing this schedule:
`[base_seed, sample_seed, 0x6D656173]` for measurement outcomes and
`[base_seed, sample_seed, 0x73637262]` for the optional global scramble.
Thus toggling `scramble` cannot move gate placement, Clifford, or measurement
candidate draws.


In [4]:
a_cfg = CircuitConfig(graph_spec='cycle', n=8, base_seed=10, total_layers_override=1)
b_cfg = CircuitConfig(graph_spec='cycle', n=8, base_seed=11, total_layers_override=1)
a_draws = CircuitBuilder(a_cfg, 2).rng.integers(2**32, size=6)
b_draws = CircuitBuilder(b_cfg, 1).rng.integers(2**32, size=6)
print('equal scalar sums:', 10 + 2 == 11 + 1)
print('pair-seeded streams differ:', not (a_draws == b_draws).all())


equal scalar sums: True
pair-seeded streams differ: True


## Warmup and measured iterators

`warmup_layers_iter()` yields gate-only prescrambling layers. `layers()` yields
the measured phase lazily, while `schedule()` materializes that phase as a list.
A literal `total_layers_override` controls the measured iterator only.


In [5]:
local = CircuitConfig(
    graph_spec='cycle', n=8, picture='purification',
    warmup_layers=2, total_layers_override=3,
)
builder = CircuitBuilder(local, 9)
warm = list(builder.warmup_layers_iter())
measured = builder.schedule()
print('warmup / measured:', len(warm), '/', len(measured))
print('warmup is gate-only:', all(layer.n_measurements == 0 for layer in warm))


warmup / measured: 2 / 3
warmup is gate-only: True


## Summary

The scheduler implements all four gate modes and all four measurement modes
with a fixed schema-v2 construction stream. Measurement outcomes and global
scrambling remain separately tagged runner streams.
